In [5]:
from dlshogi.network.policy_value_network import policy_value_network
from dlshogi import serializers
from pydlshogi2.dataloader import HcpeDataLoader
import torch
import sqlite3

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batchsize = 512
train_data = "/workspace/test2.hcpe"
arg_network = "kifcaption"
arg_model = "/workspace/model/model_resnet10_swish-072_for_caption"
model = policy_value_network(arg_network)
model.to(device)
serializers.load_npz(arg_model, model)

In [7]:
train_dataloader = HcpeDataLoader(train_data, batchsize, device, shuffle=True)

In [9]:
# SQLiteデータベースに接続
conn = sqlite3.connect("/workspace/kif_caption/comments.db")
cursor = conn.cursor()

for x1, x2, move_label, result, index in train_dataloader:
    y1, y2, policy_hidden, value_hidden, resnet_hidden = model(x1, x2)
    policy_features = torch.flatten(policy_hidden, 1)
    value_features = torch.flatten(value_hidden, 1)
    features = torch.cat([policy_features, value_features], dim=-1)
    print(features.shape)
    print(move_label[0])
    print(result.shape)
    print(index[0])

    # コメントの取得
    cursor.execute('SELECT * FROM comments WHERE comment_index = ?', (index[0].item(),))
    row = cursor.fetchone()
    break
if row:
    print(f"インデックス {row[0]} のコメント: {row[1]}")
else:
    print("コメントが見つかりません")
# 接続を閉じる
conn.close()

torch.Size([512, 4374])
tensor(49, device='cuda:0')
torch.Size([512, 1])
tensor(141065, device='cuda:0', dtype=torch.int32)
インデックス 141065 のコメント: 角のラインに入っていた飛車を浮く。△7五桂と反撃の足掛かりにできれば面白いが、先手の追及はやみそうにない。


In [6]:
import sqlite3

# SQLiteデータベースに接続
conn = sqlite3.connect("/workspace/kif_caption/comments.db")
cursor = conn.cursor()

# コメントテーブルの件数を取得
cursor.execute('SELECT COUNT(*) FROM comments')
count = cursor.fetchone()[0]  # 結果から件数を取得

print(f"データベースに登録されているコメントの件数: {count}")

# コメントの取得
cursor.execute('SELECT * FROM comments WHERE comment_index = ?', (179,))
row = cursor.fetchone()

if row:
    print(f"インデックス {row[0]} のコメント: {row[1]}")
else:
    print("コメントが見つかりません")

# 接続を閉じる
conn.close()

データベースに登録されているコメントの件数: 141379
インデックス 179 のコメント: 互いに駒に当てる手が続いた。△7六飛には▲6五角がある。△1六飛と交換を迫ってどうか。飛車が手に入れば、後手は△4七角が楽しみだ。検討の結果、△1六歩には▲1八飛△3四飛▲1六飛の順で先手が指せるようだ。以下、(1)△4五桂は▲4六銀△3八角▲4五銀△1六角成▲同香△3五飛に、▲5六角か▲5六銀。後手は持ち歩が少ないため攻めを続けづらい。後手「歩が足りていないですね。じゃあ、(68手目△4六歩と)取り込んでからダメなんですね」


In [5]:
import numpy as np
dtypeHcp = np.dtype((np.uint8, 32))
dtypeEval = np.dtype(np.int16)
dtypeMove16 = np.dtype(np.int16)
dtypeGameResult = np.dtype(np.int8)

HuffmanCodedPosAndEvalComment = np.dtype(
    [('hcp', dtypeHcp),
     ('eval', dtypeEval),
     ('bestMove16', dtypeMove16),
     ('gameResult', dtypeGameResult),
     ('dummy', np.uint8),
     ('comment_index', np.int32),
	])

def load_hcpe_file(filepath):
    # ファイルを HuffmanCodedPosAndEval の形式で読み取る
    data = np.fromfile(filepath, dtype=HuffmanCodedPosAndEvalComment)
    return data

In [17]:
# 使用例
hcpe_data = load_hcpe_file("/workspace/train1.hcpe")

# データの内容を確認
for i, record in enumerate(hcpe_data[:10]):  # 最初の10レコードを表示
    print(record["comment_index"])

0
1
2
2
3
4
5
6
7
8
